In [52]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.base import TransformerMixin,BaseEstimator
import numpy as np

In [53]:
data = pd.read_csv("customer_churn_dataset-training-master.csv")

In [56]:
data.dropna(inplace=True)

In [57]:
def add_random_nan_values(df:pd.DataFrame,cols:list,frac:float=0.02) ->pd.DataFrame:
    for col in cols:
        row_idx = df.sample(frac=frac).index
        df.loc[row_idx,col] = np.nan
    

In [58]:
add_random_nan_values(data,["Payment Delay","Usage Frequency"])

In [59]:
data.isna().sum()

CustomerID              0
Age                     0
Gender                  0
Tenure                  0
Usage Frequency      8817
Support Calls           0
Payment Delay        8817
Subscription Type       0
Contract Length         0
Total Spend             0
Last Interaction        0
Churn                   0
dtype: int64

In [65]:
class GroupedImputer(BaseEstimator,TransformerMixin):

    def __init__(self,group_col,agg_col):
        self.group_col = group_col
        self.agg_col = agg_col
    
    def fit(self,X,y=None):
        self.grouped_mean = X.groupby(self.group_col)[self.agg_col].mean()
        self.global_mean = X[self.agg_col].mean()
        return self
    
    def transform(self,X,y=None):
        X = X.copy()
        fill_values = X[self.group_col].map(self.grouped_mean).fillna(self.global_mean)
        X[self.agg_col] = X[self.agg_col].fillna(fill_values)
        return(X)


In [66]:
data.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0
1,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0
2,4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.0,3.0,1.0
3,5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.0,29.0,1.0
4,6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.0,20.0,1.0


In [67]:
X = data.drop(columns='Churn')
y = data["Churn"]

X.shape, y.shape

((440832, 11), (440832,))

In [69]:

X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size=0.15,
                                                    stratify=y,
                                                    random_state=12)
imputer = GroupedImputer(group_col = 'Gender', agg_col = "Usage Frequency")
imputer.fit_transform(X_train).isna().sum()

CustomerID              0
Age                     0
Gender                  0
Tenure                  0
Usage Frequency         0
Support Calls           0
Payment Delay        7512
Subscription Type       0
Contract Length         0
Total Spend             0
Last Interaction        0
dtype: int64

In [98]:
class GroupedImputerFull(TransformerMixin,BaseEstimator):

    def __init__(self,group_col):
        self.group_col = group_col
    
    def fit(self,X,y=None):
        self.na_cols_ = X.columns[X.isna().any()].to_list()
        self.grouped_means = X.groupby(self.group_col)[self.na_cols_].mean()
        self.global_means = X[self.na_cols_].mean()
        return self
    
    def transform(self,X,y=None):
        X = X.copy()
        for col in self.na_cols_:
            fill_values = X[self.group_col].map(self.grouped_means[col]).fillna(self.global_means)
            X[col] = X[col].fillna(fill_values)
        return X
            

In [101]:
tranf = GroupedImputerFull("Gender")
tranf.fit_transform(X_train).isna().sum()

CustomerID           0
Age                  0
Gender               0
Tenure               0
Usage Frequency      0
Support Calls        0
Payment Delay        0
Subscription Type    0
Contract Length      0
Total Spend          0
Last Interaction     0
dtype: int64

In [96]:
num_cols =data.select_dtypes(include=[int,float]).columns.to_list()[1:]
low = data[num_cols].quantile(0.25)
high = data[num_cols].quantile(0.75)
np.clip(data[num_cols],low,high,axis=1)

,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Churn
0,30.0,39.0,14.0,5.0,18.0,830.00,17.0,1.0
1,48.0,46.0,9.0,6.0,8.0,557.00,7.0,1.0
2,48.0,16.0,9.0,6.0,18.0,480.00,7.0,1.0
3,48.0,38.0,21.0,6.0,7.0,480.00,22.0,1.0
4,29.0,32.0,20.0,5.0,8.0,617.00,20.0,1.0
...,...,...,...,...,...,...,...,...
440828,42.0,46.0,15.0,1.0,6.0,716.38,8.0,0.0
440829,29.0,16.0,13.0,1.0,19.0,745.38,7.0,0.0
440830,29.0,35.0,23.0,1.0,6.0,830.00,9.0,0.0
440831,29.0,46.0,14.0,2.0,6.0,602.55,7.0,0.0


In [ ]:
np.clip()

Age                  29.0
Tenure               16.0
Usage Frequency       9.0
Support Calls         1.0
Payment Delay         6.0
Total Spend         480.0
Last Interaction      7.0
Churn                 0.0
Name: 0.25, dtype: float64